## Averaging Tangents and using area to prevent large/small squares

* I removed the `smoothing_size` parameter from our patching algorithm by calculating multiple tangents on the contour and taking the average 
* Taking this idea from Ryan, I prevent too large/small patches from forming by taking the area of the patch and preventing patches of a certain percentage from forming. 

* STILL WORK-IN-PROGRESS: trying to figure out how to handle patches that straddle two distinct regions of the epithelium. To get around this, I introduce two new functions: `sample_line_pixels` and `epithelium_to_background`. `sample_line_pixel` gets the pixel values along a start and end point. It is used to find the pixel values along the four sides of each patch. `eptihelium_to_background` checks to see whether these sides of the patches flag a pattern that reveals that a patch is straddling two distinct regions. This pattern is epithelium -> non-epithelium -> epithelium. 
    * If this pattern is flagged, that patch is removed

*** EDIT *** I am pushing the first iteration of the logic, which for some reason does not work and flags too many patches as having this pattern. You can find the debugging attempt near the bottom. 

In [1]:
extension_factor = 1.1; 
step_size = 2;
max_width = 2048
overlap_threshold = 0.3
weight_epithelium_coverage = 0.8
weight_background_pixels = 0.2

slice_name = "h1849462  h&e_ROI_3.tif"
slice_detail = "h&e_benign_case85_match1"
mask_name = slice_detail + ".png"
patched_slice_name = "output/patched_" + mask_name
metrics_file_name = "output/metrics_before_improvement_" + slice_detail + ".csv"
patch_length_filename = "output/patch_lengths_before_improvement" + slice_detail + ".csv"

In [2]:
# Loading libraries and functions

import numpy as np
import cv2
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
from matplotlib import colors
import pandas as pd
import csv

# Function to draw patches
def draw_patches(image, patch_list, color = [255,0,0], thickness = 20):
    patches_on_slice = image.copy()
    for contour_patches in patch_list:
        for patch in contour_patches:
            points = patch.astype(np.int32)
            cv2.polylines(patches_on_slice, [points], isClosed = True, color = color, thickness = thickness)        
    return patches_on_slice

# Function to calculate patch overlap
def patch_overlap(square1,square2):

    square1 = Polygon(square1)
    square2 = Polygon(square2)

    if not square1.intersects(square2):
        return 0
    
    intersection_area = square1.intersection(square2).area
    smallest_area = min(square1.area,square2.area)
    return intersection_area/smallest_area

def sample_line_pixels(start, end, mask):
    x1, y1 = map(int, start)
    x2, y2 = map(int, end)

    # Get points along the line
    num_points = max(abs(x2 - x1), abs(y2 - y1), 1)  
    x_values = np.linspace(x1, x2, num_points).astype(int)
    y_values = np.linspace(y1, y2, num_points).astype(int)

    x_values = np.clip(x_values, 0, mask.shape[1] - 1)
    y_values = np.clip(y_values, 0, mask.shape[0] - 1)

    return mask[y_values, x_values]

def epithelium_background_epithelium_transition(pixel_values, min_group_size=10):
    is_epithelium = (pixel_values > 0).astype(int)
    transitions = np.diff(is_epithelium)
    
    change_indices = np.where(transitions != 0)[0] + 1 
    unique_groups = np.split(pixel_values, change_indices)

    # Remove very small groups (noise)
    filtered_groups = [group for group in unique_groups if len(group) >= min_group_size]

    return len(filtered_groups) == 3



# Reading data
tissue_slice = cv2.cvtColor(cv2.imread("cases/" + slice_name), cv2.COLOR_BGR2RGB)
epithelium_mask = cv2.cvtColor(cv2.imread("cases/" + mask_name), cv2.COLOR_BGR2RGB)
epithelium_mask_gray = cv2.cvtColor(epithelium_mask, cv2.COLOR_RGB2GRAY)
Otsu_threshold, epithelium_mask_2D = cv2.threshold(epithelium_mask_gray, 0, 255, 
                            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Getting contours
contours, hierarchy = cv2.findContours(epithelium_mask_2D, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Getting tangents and normals on contour points
all_normals = []
all_tangents = []

# Predefine the smoothing parameter sizes
parameter_sizes = np.array([75, 100, 125, 150, 200])

for contour in contours:
    contour_points = contour.reshape(-1, 2)
    num_contour_points = len(contour_points)

    # Precompute indices for faster access
    prev_indices = (np.arange(num_contour_points)[:, None] - parameter_sizes) % num_contour_points
    next_indices = (np.arange(num_contour_points)[:, None] + parameter_sizes) % num_contour_points

    # Extract the actual contour points
    prev_points = contour_points[prev_indices]
    next_points = contour_points[next_indices]

    # Compute tangents
    tangents = next_points - prev_points
    tangents = tangents / (np.linalg.norm(tangents, axis=2, keepdims=True) + 1e-8)  

    # Take the mean along the smoothing dimension
    avg_tangents = np.mean(tangents, axis=1)

    # Normalize again
    avg_tangents = avg_tangents / (np.linalg.norm(avg_tangents, axis=1, keepdims=True) + 1e-8)

    # Compute normals
    normals = np.zeros_like(avg_tangents)
    normals[:, 0] = -avg_tangents[:, 1]
    normals[:, 1] = avg_tangents[:, 0]

    all_normals.append(normals)
    all_tangents.append(avg_tangents)
    
all_patches = []
all_patch_lengths = []
skipped_points = []
height, width = epithelium_mask_2D.shape

# Finding patch corners on each contour
for c in range(len(contours)):
    contour_points = contours[c].reshape(-1, 2)
    contour_normals = all_normals[c]
    contour_tangents = all_tangents[c]
    contour_patch_lengths = []
    squares = []

    contour_area = cv2.contourArea(contours[c])
    side_length = np.sqrt(contour_area)
    min_patch_length = side_length * 0.05  # squares at least 5% of area 
    max_patch_length = side_length * 0.50
    
    for i, point in enumerate(contour_points):
        point = point.astype(int)

        steps = np.arange(0, max_width, step_size)[:, None]
        normal_points = np.clip(point - steps * contour_normals[i], 0, [width - 1, height - 1]).astype(int)
    
        pixel_values_normal = epithelium_mask_2D[normal_points[:, 1], normal_points[:, 0]]

        stop = np.where(pixel_values_normal == 0)[0]
        epithelium_width = stop[0] * step_size if stop.size > 0 else max_width

        patch_length = int(epithelium_width * extension_factor)
        half_patch_length = patch_length // 2 

        if patch_length < min_patch_length or patch_length > max_patch_length:
            continue
        
        corners = np.array([
            point + (contour_tangents[i] * half_patch_length),
            point - (contour_tangents[i] * half_patch_length),
            point - (contour_tangents[i] * half_patch_length) - (contour_normals[i] * patch_length),
            point + (contour_tangents[i] * half_patch_length) - (contour_normals[i] * patch_length)
        ])

        side_pixel_values = [
            sample_line_pixels(corners[0], corners[1], epithelium_mask_2D),
            sample_line_pixels(corners[1], corners[2], epithelium_mask_2D),
            sample_line_pixels(corners[2], corners[3], epithelium_mask_2D),
            sample_line_pixels(corners[3], corners[0], epithelium_mask_2D)
        ]

        if any(epithelium_background_epithelium_transition(side_values) for side_values in side_pixel_values):
            skipped_points.append(tuple(point))
            print(f"Skipping patch at {point} due to epithelium-background-epithelium transition.")
            continue


        squares.append(corners)
        contour_patch_lengths.append(patch_length)

all_selected_patches = []
all_selected_patch_lengths = []

# Removing highly overlapping contours
for c in range(len(contours)):
    contour_points = contours[c].reshape(-1, 2)

    if c >= len(all_patches) or len(all_patches[c]) == 0:
        continue
    
    selected_patches = []
    selected_patch_lengths = []
    for i, point in enumerate(contour_points):
        if i >= len(all_patches[c]):  # Ensure index is in range
            continue

        current_patch = all_patches[c][i]
        current_patch_length = all_patch_lengths[c][i]
        if not any(patch_overlap(current_patch, ith_selected_patch) > overlap_threshold \
                 for ith_selected_patch in selected_patches):
            selected_patches.append(current_patch)
            selected_patch_lengths.append(current_patch_length)
    all_selected_patches.append(selected_patches)
    all_selected_patch_lengths.append(selected_patch_lengths)

selected_patches_on_slice = draw_patches(tissue_slice, all_selected_patches)

# Exporting patched slice
cv2.imwrite(patched_slice_name, cv2.cvtColor(selected_patches_on_slice, cv2.COLOR_RGB2BGR))

# Calculating performance metrics
total_epithelium_pixels = (epithelium_mask_2D > 0).sum()
patch_mask = np.zeros_like(epithelium_mask_2D, dtype=np.uint8)

#For all patches in all contours, use them to create a mask
for contour_patches in all_selected_patches:
    for patch in contour_patches:
        patch = np.array(patch,dtype = np.int32).reshape(-1, 1, 2)
        cv2.fillPoly(patch_mask, [patch], 255)

number_of_patch_mask_pixels = np.count_nonzero(patch_mask > 0)

covered_epithelium_pixels = np.count_nonzero((patch_mask > 0) & (epithelium_mask_2D > 0))
covered_background_pixels = np.count_nonzero((patch_mask > 0) & (epithelium_mask_2D == 0))

total_epithelium_coverage = (covered_epithelium_pixels / total_epithelium_pixels*100)
background_pixel_percent = (covered_background_pixels / number_of_patch_mask_pixels*100)
score = weight_epithelium_coverage*total_epithelium_coverage + weight_background_pixels*(100 - background_pixel_percent)

# Exporting performance metrics and parameter values
metrics = pd.DataFrame({'Metric':['total_epithelium_coverage', 'background_pixel_percent',
                       'score', 'extension_factor', 'step_size',
                       'max_width', 'overlap_threshold'], 
              'Value':[total_epithelium_coverage, background_pixel_percent, score,
               extension_factor, step_size, max_width, overlap_threshold]})

metrics.to_csv(metrics_file_name, index=False)

# Exporting patch lengths
max_length = max(len(contour_patch_lengths) for contour_patch_lengths in all_selected_patch_lengths)

padded_list = [contour_patch_lengths + [''] * (max_length - len(contour_patch_lengths)) \
               for contour_patch_lengths in all_selected_patch_lengths]

transposed_list = list(map(list, zip(*padded_list)))

with open(patch_length_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(transposed_list)

plt.imshow(selected_patches_on_slice);

Skipping patch at [1752 2803] due to epithelium-background-epithelium transition.
Skipping patch at [1749 2817] due to epithelium-background-epithelium transition.
Skipping patch at [1747 2819] due to epithelium-background-epithelium transition.
Skipping patch at [1728 2824] due to epithelium-background-epithelium transition.
Skipping patch at [1725 2824] due to epithelium-background-epithelium transition.
Skipping patch at [1724 2823] due to epithelium-background-epithelium transition.
Skipping patch at [1723 2823] due to epithelium-background-epithelium transition.
Skipping patch at [1722 2822] due to epithelium-background-epithelium transition.
Skipping patch at [1686 2822] due to epithelium-background-epithelium transition.
Skipping patch at [1685 2823] due to epithelium-background-epithelium transition.
Skipping patch at [1667 2823] due to epithelium-background-epithelium transition.
Skipping patch at [1666 2824] due to epithelium-background-epithelium transition.
Skipping patch a

ZeroDivisionError: division by zero

# TRYING TO DEBUG

In [3]:
def epithelium_background_epithelium_transition(pixel_values):
    is_epithelium = (pixel_values > 0).astype(int)
    

    transitions = np.diff(is_epithelium)
    
    change_indices = np.where(transitions != 0)[0] + 1  

    unique_groups = np.split(pixel_values, change_indices)

    print(f"Groups found: {[list(group)[:5] for group in unique_groups]} ...") 
    

    if len(unique_groups) == 3:
        print("❌ Epithelium → Background → Epithelium detected! Rejecting patch.")
        return True  
    
    print("✅ No transition found. Patch is safe.")
    return False 


In [4]:
for j, side in enumerate(side_pixel_values):
    print(f"Side {j} pixel values: {side}")  # Debugging line
    if epithelium_background_epithelium_transition(side):
        print(f"Rejected Patch at {point}: Transition Detected on Side {j}")
        skipped_points.append(tuple(point))
        break
else:
    squares.append(corners)
    contour_patch_lengths.append(patch_length)

Side 0 pixel values: [255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255
 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255 255   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0  

In [20]:
print(f"Total Coverage: {total_epithelium_coverage:.2f}%")
print(f"Background Pixel Percentage: {background_pixel_percent:.2f}%")
print(f"Score Percentage: {score:.2f}%")

Total Coverage: 94.63%
Background Pixel Percentage: 17.12%
Score Percentage: 92.28%
